# Планирование рабочих смен — исправленная версия

### Исправленные баги относительно исходного `распределение.ipynb`

1. **`max_cover` — мягкое ограничение (было жёстким)**  
   Исходный код содержал `lpSum(covering) <= required + 2` как жёсткое (hard) ограничение.  
   Это блокировало назначение сотрудников с длинными сменами, т.к. в одни часы они создавали «избыток», даже если в другие часы их присутствие критично.  
   Исправлено: введена переменная `surplus[(d,h,s)]` с умеренным штрафом — превышение допускается, но штрафуется.

2. **Некорректная проверка статуса решателя**  
   Исходный код: `if LpStatus[prob.status] in ('Optimal', 'Not Solved')` — `'Not Solved'` означает, что задача **вообще не решалась**, а не что найдено приближённое решение.  
   Исправлено: `if status_str in ('Optimal', 'Feasible')` — включает допустимое решение при срабатывании `timeLimit`.

3. **O(n³) пересчёт covering-списков → O(1) через индекс**  
   Исходный код заново перебирал все `valid_shifts` в тройном цикле `(d, h, s)` при построении ограничений.  
   Исправлено: `covering_index[(d, h, s)]` строится один раз при создании переменных.

4. **Некорректный диапазон `t_max_start`**  
   Исходный код: `min(23 - L, end_ok - L)` корректен, но без явного ограничения `t_start + L <= min(23, end_ok)`.  
   Теперь явно проверяется что `finishtime = t_start + L` укладывается и в 23:00 и в `end_ok` сотрудника.

In [3]:
!pip install highspy pulp

import pandas as pd
import numpy as np
from pulp import *
from datetime import datetime, timedelta
from collections import defaultdict

# ======================
# 1. Загрузка данных
# ======================
reqlabor_df      = pd.read_csv('reqlabor.csv')
sched_df         = pd.read_csv('sched.csv')
shifts_df        = pd.read_csv('shifts.csv')
station_prior_df = pd.read_csv('station_priorities.csv')
staff_limits_df  = pd.read_csv('staff_limits.csv')

stations = ['BVR', 'C', 'FF', 'K', 'TS']

# ======================
# 2. Загрузка прогноза гостей
# ======================
forecast_df = pd.read_csv('forecast_2026-04-27_2026-05-03.csv')
forecast_df.rename(columns={'predicted_guests': 'guests_count'}, inplace=True)
forecast_df['sale_date'] = pd.to_datetime(forecast_df['sale_date'])

BASE_DATE = forecast_df['sale_date'].min()
print(f"Базовая дата: {BASE_DATE.date()}, день недели: {BASE_DATE.day_name()}")

# ======================
# 3. Функция потребности по станциям
# ======================
reqlabor_df['version'] = reqlabor_df['version'].str.strip()

_reqlabor_index = defaultdict(list)
for _, row in reqlabor_df.iterrows():
    _reqlabor_index[(row['station_key'], row['version'])].append(
        (row['guests_count'], row['reqlabor'])
    )
for k in _reqlabor_index:
    _reqlabor_index[k].sort(key=lambda x: x[0])


def get_version(weekday_0based, hour):
    if weekday_0based < 5:
        return 'будни/утр.' if hour < 10 else 'будни/осн.'
    else:
        return 'вых/утр.' if hour < 10 else 'вых/осн.'


def required_labor(station, version, guests):
    pairs = _reqlabor_index[(station, version)]
    if not pairs:
        return 0
    for gc, rl in pairs:
        if guests <= gc:
            return rl
    return pairs[-1][1]


req = {}
for _, row in forecast_df.iterrows():
    d = row['sale_date'].weekday()
    h = row['sale_hour']
    guests = row['guests_count']
    version = get_version(d, h)
    for s in stations:
        req[(d, h, s)] = required_labor(s, version, guests)

print("Матрица потребности построена.")

# ======================
# 4. Подготовка данных сотрудников
# ======================
avail = defaultdict(dict)
for _, row in sched_df.iterrows():
    e = int(row['employee_id'])
    d = int(row['day']) - 1
    avail[e][d] = (int(row['starttime']), int(row['finishtime']))

worktime_limit = {}
shift_limit    = {}
for _, row in staff_limits_df.iterrows():
    e = int(row['employee_id'])
    worktime_limit[e] = int(row['worktime_limit'])
    shift_limit[e]    = int(row['shift_limit'])

station_prior = defaultdict(dict)
for _, row in station_prior_df.iterrows():
    station_prior[int(row['employee_id'])][row['station_key']] = int(row['station_priority'])

shift_prior    = dict(zip(shifts_df['shift_duration'].astype(int),
                          shifts_df['shift_priority'].astype(int)))
allowed_lengths = sorted(shift_prior.keys())

employees = sorted(staff_limits_df['employee_id'].astype(int).unique())
print(f"Сотрудников: {len(employees)}")

# ======================
# 5. Построение MILP-модели
# ======================
prob = LpProblem("Staff_Scheduling", LpMinimize)

x = {}
valid_shifts = []
covering_index = defaultdict(list)

for e in employees:
    for d in range(7):
        if d not in avail[e]:
            continue
        start_ok, end_ok = avail[e][d]
        max_L = min(shift_limit[e], 9)

        for L in allowed_lengths:
            if L > max_L:
                continue
            t_min_start = max(7, start_ok)
            t_max_start = min(23 - L, end_ok - L)
            if t_min_start > t_max_start:
                continue

            for t_start in range(t_min_start, t_max_start + 1):
                for s in stations:
                    key = (e, d, t_start, L, s)
                    var = LpVariable(f"x_{e}_{d}_{t_start}_{L}_{s}", cat='Binary')
                    x[key] = var
                    valid_shifts.append((*key, var))
                    for h in range(t_start, t_start + L):
                        if 7 <= h < 23:
                            covering_index[(d, h, s)].append(var)

print(f"Переменных решений: {len(x)}")

# --- Одна смена в день на сотрудника
for e in employees:
    for d in range(7):
        vars_day = [v for (ee, dd, ts, L, s, v) in valid_shifts if ee == e and dd == d]
        if vars_day:
            prob += lpSum(vars_day) <= 1, f"one_shift_{e}_{d}"

# --- Переменные дефицита и профицита
shortage = {}
surplus   = {}

for d in range(7):
    for h in range(7, 23):
        for s in stations:
            req_val = req.get((d, h, s), 0)
            if req_val > 0 or covering_index[(d, h, s)]:
                shortage[(d, h, s)] = LpVariable(f"sh_{d}_{h}_{s}", lowBound=0, cat='Integer')
                surplus[(d, h, s)]  = LpVariable(f"su_{d}_{h}_{s}", lowBound=0, cat='Integer')

# --- Ограничение "всегда минимум 1 человек на станции" (НОВОЕ)
for d in range(7):
    for h in range(7, 23):
        for s in stations:
            if covering_index[(d, h, s)]:
                prob += lpSum(covering_index[(d, h, s)]) >= 1, f"min_one_{d}_{h}_{s}"

# --- Ограничения покрытия (с учётом минимума уже выше)
for d in range(7):
    for h in range(7, 23):
        for s in stations:
            req_val  = req.get((d, h, s), 0)
            cvars    = covering_index[(d, h, s)]
            sh       = shortage.get((d, h, s))
            su       = surplus.get((d, h, s))
            if sh is None:
                continue
            prob += lpSum(cvars) + sh >= req_val, f"min_cover_{d}_{h}_{s}"
            prob += lpSum(cvars) - su <= req_val + 2, f"max_cover_{d}_{h}_{s}"

# --- Недельные часы
for e in employees:
    emp_vars = [(L, v) for (ee, dd, ts, L, s, v) in valid_shifts if ee == e]
    if emp_vars:
        prob += lpSum(L * v for L, v in emp_vars) <= worktime_limit[e], f"weekly_h_{e}"

# --- Выходные дни: не более 5 рабочих, не менее 1 рабочего
day_work = {}
for e in employees:
    for d in range(7):
        vars_day = [v for (ee, dd, ts, L, s, v) in valid_shifts if ee == e and dd == d]
        if vars_day:
            y = LpVariable(f"dw_{e}_{d}", cat='Binary')
            prob += lpSum(vars_day) <= y
            prob += lpSum(vars_day) >= y
        else:
            y = LpVariable(f"dw_{e}_{d}", cat='Binary')
            prob += y == 0
        day_work[(e, d)] = y

    prob += lpSum(day_work[(e, d)] for d in range(7)) <= 5, f"min_rest_{e}"
    prob += lpSum(day_work[(e, d)] for d in range(7)) >= 1, f"min_work_{e}"

# --- Целевая функция
BIG_PENALTY     = 100_000
OVERAGE_PENALTY = 500

objective = []
for var in shortage.values():
    objective.append(BIG_PENALTY * var)
for var in surplus.values():
    objective.append(OVERAGE_PENALTY * var)
for (e, d, ts, L, s, var) in valid_shifts:
    cost = shift_prior[L] + station_prior[e].get(s, 2)
    # Разрыв симметрии: мельчайшая добавка, пропорциональная ID сотрудника
    cost += e * 1e-5
    objective.append(cost * var)

prob += lpSum(objective), "total_cost"

# ======================
# 6. Решение
# ======================
print("Запуск решателя HiGHS...")
prob.solve(HiGHS(
    msg=True,
    timeLimit=300,          # можно оставить 600, если хотите дольше
    mip_rel_gap=0,       # остановка при зазоре 2%
    threads=4,              # если ЦП поддерживает
    presolve='on'
))

status_str = LpStatus[prob.status]
print(f"Статус: {status_str}")

total_shortage = sum(int(round(v.varValue or 0)) for v in shortage.values())
total_surplus  = sum(int(round(v.varValue or 0)) for v in surplus.values())
print(f"Суммарный дефицит (чел-часы): {total_shortage}")
print(f"Суммарный избыток > req+2 (чел-часы): {total_surplus}")

if total_shortage > 0:
    print("\nДетали дефицита (физически неустранимые при данных ограничениях):")
    for (d, h, s), var in shortage.items():
        val = int(round(var.varValue or 0))
        if val > 0:
            date_str = (BASE_DATE + timedelta(days=d)).strftime('%Y-%m-%d')
            print(f"  {date_str} час {h} станция {s}: нехватка {val}")

# ======================
# 7. Сохранение расписания
# ======================
schedule = []
if status_str in ('Optimal', 'Feasible'):
    for (e, d, ts, L, s, var) in valid_shifts:
        if var.varValue is not None and var.varValue > 0.5:
            ds_date = (BASE_DATE + timedelta(days=d)).strftime('%Y-%m-%d')
            schedule.append({
                'ds':          ds_date,
                'station_key': s,
                'employee_id': e,
                'starttime':   ts,
                'finishtime':  ts + L,
            })

    schedule_df = pd.DataFrame(schedule,
                               columns=['ds', 'station_key', 'employee_id', 'starttime', 'finishtime'])
    schedule_df.sort_values(['ds', 'station_key', 'starttime'], inplace=True)

    schedule_df.to_excel('schedule_result.xlsx', index=False)
    print(f"\nРасписание сохранено в schedule_result.xlsx")
    print(f"Строк: {len(schedule_df)}, уникальных сотрудников: {schedule_df['employee_id'].nunique()}")
    print(schedule_df.head(20).to_string())
else:
    print(f"Решение не найдено. Статус: {status_str}")

forecast_df.to_excel('forecast_guests.xlsx', index=False)
print("Прогноз сохранён в forecast_guests.xlsx")

Базовая дата: 2026-04-27, день недели: Monday
Матрица потребности построена.
Сотрудников: 69
Переменных решений: 57410
Запуск решателя HiGHS...
Статус: Optimal
Суммарный дефицит (чел-часы): 0
Суммарный избыток > req+2 (чел-часы): 0

Расписание сохранено в schedule_result.xlsx
Строк: 194, уникальных сотрудников: 69
             ds station_key  employee_id  starttime  finishtime
132  2026-04-27         BVR           43          7          16
11   2026-04-27         BVR            3         16          23
1    2026-04-27           C            1          7          16
26   2026-04-27           C            6          7          14
182  2026-04-27           C           61         10          18
123  2026-04-27           C           38         12          19
142  2026-04-27           C           45         12          15
105  2026-04-27           C           32         15          23
96   2026-04-27           C           30         17          23
128  2026-04-27           C           39    